In [26]:
import os

os.environ["LOCAL_BASE_URL"] = "http://202.31.200.88:8111"
os.environ["LOCAL_MODEL_NAME"] = "openai/gpt-oss-20b"
os.environ["LOCAL_API_KEY"] = "not_used"

print("환경 변수 설정 완료")

환경 변수 설정 완료


In [27]:
from langchain_core.documents import Document

raw_text = """
부영그룹은 2021년 이후 태어난 직원 자녀 1인당 1억원의 출산장려금을 지원한다고 밝혔다.
연년생 또는 쌍둥이의 경우 지원금은 합산되며, 셋째 출산 시 주택 제공 방안도 언급했다.
정부와 지자체 또한 부모급여 인상, 첫만남이용권, 아동수당, 지자체 현금 지원 등 저출생 대책을 확대하고 있다.
""".strip()

docs = [Document(page_content=raw_text, metadata={"source": "synthetic_buyeong_policy"})]
print("문서 개수:", len(docs))
print("문서 샘플:", docs[0].page_content)

문서 개수: 1
문서 샘플: 부영그룹은 2021년 이후 태어난 직원 자녀 1인당 1억원의 출산장려금을 지원한다고 밝혔다.
연년생 또는 쌍둥이의 경우 지원금은 합산되며, 셋째 출산 시 주택 제공 방안도 언급했다.
정부와 지자체 또한 부모급여 인상, 첫만남이용권, 아동수당, 지자체 현금 지원 등 저출생 대책을 확대하고 있다.


In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
splits = splitter.split_documents(docs)

print("분할 개수:", len(splits))
for i, d in enumerate(splits[:3], 1):
    print(f"[{i}]", d.page_content)

분할 개수: 2
[1] 부영그룹은 2021년 이후 태어난 직원 자녀 1인당 1억원의 출산장려금을 지원한다고 밝혔다.
연년생 또는 쌍둥이의 경우 지원금은 합산되며, 셋째 출산 시 주택 제공 방안도 언급했다.
[2] 정부와 지자체 또한 부모급여 인상, 첫만남이용권, 아동수당, 지자체 현금 지원 등 저출생 대책을 확대하고 있다.


In [29]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = FAISS.from_documents(splits, embedding_model)

# 저장/로드 예시
index_dir = "./faiss_index_demo"
vectorstore.save_local(index_dir)
loaded_vectorstore = FAISS.load_local(index_dir, embedding_model, allow_dangerous_deserialization=True)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 21045.26it/s]


In [30]:
question = "부영그룹의 출산 장려 정책에 대해 설명해주세요."

retriever = loaded_vectorstore.as_retriever(search_kwargs={"k": 2})
retrieved_docs = retriever.invoke(question)
print("retriever 결과 개수:", len(retrieved_docs))
for i, d in enumerate(retrieved_docs, 1):
    print(f"[{i}]", d.page_content)

print("\n유사도 score 확인:")
for i, (d, score) in enumerate(loaded_vectorstore.similarity_search_with_score(question, k=2), 1):
    print(f"[{i}] score={score:.4f} |", d.page_content)

retriever 결과 개수: 2
[1] 부영그룹은 2021년 이후 태어난 직원 자녀 1인당 1억원의 출산장려금을 지원한다고 밝혔다.
연년생 또는 쌍둥이의 경우 지원금은 합산되며, 셋째 출산 시 주택 제공 방안도 언급했다.
[2] 정부와 지자체 또한 부모급여 인상, 첫만남이용권, 아동수당, 지자체 현금 지원 등 저출생 대책을 확대하고 있다.

유사도 score 확인:
[1] score=0.5565 | 부영그룹은 2021년 이후 태어난 직원 자녀 1인당 1억원의 출산장려금을 지원한다고 밝혔다.
연년생 또는 쌍둥이의 경우 지원금은 합산되며, 셋째 출산 시 주택 제공 방안도 언급했다.
[2] score=1.0551 | 정부와 지자체 또한 부모급여 인상, 첫만남이용권, 아동수당, 지자체 현금 지원 등 저출생 대책을 확대하고 있다.


In [31]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 질문-답변 도우미다. 반드시 제공된 context 안에서만 답해라. 모르면 모른다고 말해라."),
    ("human", "질문: {question}\n\n문맥: {context}\n\n답변:")
])

In [32]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model=os.environ.get("LOCAL_MODEL_NAME", "qwen3-8b"),
    base_url=os.environ.get("LOCAL_BASE_URL", "http://202.31.200.130:8001/v1"),
    api_key=os.environ.get("LOCAL_API_KEY", "not_used"),
    temperature=0,
)

def format_docs(ds):
    return "\n\n".join(d.page_content for d in ds)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [33]:
answer = rag_chain.invoke("부영그룹의 출산 장려 정책에 대해 설명해주세요.")
print("반환 타입:", type(answer))
print("응답 본문:\n", answer)

APITimeoutError: Request timed out.

In [51]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from tqdm.auto import tqdm
import os

os.environ["LOCAL_BASE_URL"] = "http://202.31.200.88:8111"
os.environ["LOCAL_MODEL_NAME"] = "openai/gpt-oss-20b"
os.environ["LOCAL_API_KEY"] = "not_used"

loader = PyPDFLoader("2502.12911v3.pdf")
docs = loader.load()
print(f"문서의 수: {len(docs)}")

page_idx = 10
if len(docs) > page_idx:
    print(f"\n[페이지내용]\n{docs[page_idx].page_content[:500]}")
    print(f"\n[metadata]\n{docs[page_idx].metadata}\n")
else:
    print(f"페이지 수 부족: 총 {len(docs)}페이지")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)

splits = text_splitter.split_documents(docs)
print(f"분할된 문서의 수: {len(splits)}")
# 단계 3: 임베딩 & 벡터스토어 생성(Create Vectorstore)
# 벡터스토어를 생성합니다.
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
    show_progress=True,
)

vectorstore = FAISS.from_documents(splits, embedding_model)
print("벡터스토어 생성 완료")
# 단계 4: 검색(Search)
# 뉴스에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

def format_docs(ds):
    # 검색한 문서 결과를 하나의 문단으로 합쳐줍니다.
    return "\n\n".join(doc.page_content for doc in ds)
print("retriever 생성 완료")

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from tqdm.auto import tqdm

os.environ["OPENAI_API_BASE"] = "http://202.31.200.88:8111/v1"
os.environ["OPENAI_API_KEY"] = "not_used" model="openai/gpt-oss-20b",

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 질문-답변 도우미다. 반드시 제공된 context 안에서만 답해라. 모르면 모른다고 말해라."),
    ("human", "질문: {question}\n\n문맥: {context}\n\n답변:")
])
print("프롬프트 입력 변수: context, question")
# 단계 7: 체인 생성(Create Chain)
llm = ChatOpenAI(
    model=os.environ.get("LOCAL_MODEL_NAME", "openai/gpt-oss-20b"),
    base_url=os.environ.get("LOCAL_BASE_URL", "http://202.31.200.88:8111"),
    api_key=os.environ.get("LOCAL_API_KEY", "not_used"),
    temperature=0,
)

def format_docs(ds):
    return "\n\n".join(d.page_content for d in ds)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("rag_chain 생성 완료")
# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.

answer = rag_chain.invoke("explane KaSLA and its advantages and disadvantages.")
print("반환 타입:", type(answer))
print("응답 본문:\n", answer)

[디버깅] retriever 상태 확인:
✓ retriever 존재: <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


[디버깅] retriever 상태 확인:
✓ retriever 존재: <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.95it/s]

[디버깅] retriever 상태 확인:
✓ retriever 존재: <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.95it/s]

...

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.95it/s]

✓ retriever 동작 확인: 4개 문서 반환

프롬프트 입력 변수: context, question


rag_chain 생성 완료

[질문 실행]


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.28it/s]
DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-64f68a34-6085-46b1-82db-a36abad059de', 'content': None, 'json_data': {'messages': [{'content': '너는 질문-답변 도우미다. 반드시 제공된 context 안에서만 답해라. 모르면 모른다고 말해라.', 'role': 'system'}, {'content': '질문: explane KaSLA and its advantages and disadvantages.\n\n문맥: results highlight KaSLA’s strong robustness and effectiveness\nwhen applied directly to new, unseen scenarios.\n2) Performance on Multi-join Instances:To assess KaSLA’s\neffectiveness on complex schema linking scenarios, we evaluate\nits performance on BIRD-dev by grouping instances according\nto the number of tables needed in each instance, as shown in Ta-\nble VII. Across all groups, including single-table, two-table, and\nmulti-join (three or more tables) queries, KaSLA consistently\nachieves

반환 타입: <class 'langchain_core.messages.base.TextAccessor'>
응답 본문:
 **KaSLA (Knapsack‑Optimization‑Based Schema Linking Approach)**  
KaSLA is a schema‑linking framework designed for text‑to‑SQL systems.  
It tackles the classic “missing‑vs‑redundant” dilemma that many
generative or recall‑based methods face by formulating the linking
problem as a **knapsack optimization** task:

1. **Hierarchical linking** – first the model selects the tables that
   are relevant to the user query, then it links columns only inside
   those tables.  
2. **Knapsack optimization** – for each step the model chooses a set
   of elements (tables or columns) that maximizes a score while
   respecting a *redundancy tolerance* budget.  
3. **Binary + probabilistic scoring** – a binary scoring model
   confirms whether an element should be linked, and a probabilistic
   model estimates the likelihood of each candidate.  
4. **Redundancy estimation** – the framework explicitly estimates how
   many redundant ele